In [1]:
!pip install streamlink ultralytics
!apt-get install ffmpeg -y|

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 105.4 MB/s eta 0:00:0000:01
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [ ]:
import cv2
import time
import threading
import os
import torch
import streamlink
import sys
import atexit
import numpy as np
from ultralytics import YOLO

# --- GPU VALIDATION ---
if not torch.cuda.is_available():
    print("FATAL: This script requires a CUDA-compatible GPU.")
    sys.exit(1)

# --- CONFIGURATION ---
URLS = [
    "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", 
    "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1"
]
MODEL_NAME = 'yolo11m.pt'
DEVICE = 'cuda:0'
TARGET_CLASSES = [0]
SEGMENT_DURATION = 300
FPS = 10.0

CAM_CONFIG = {
    0: {
        "res": (955, 542),
        "zone": np.array([[126, 324], [236, 319], [301, 391], [123, 419]], dtype=np.int32)
    },
    1: {
        "res": (955, 537),
        "zone": np.array([[520, 365], [772, 466], [886, 326], [689, 262]], dtype=np.int32)
    }
}

gpu_lock = threading.Lock()
stop_event = threading.Event()
video_writers = {}

def cleanup():
    if stop_event.is_set(): return
    stop_event.set()
    time.sleep(1)
    for cid, wr in video_writers.items():
        if wr: wr.release()
    torch.cuda.empty_cache()
    print("\n[FINISH] Monitoring halted. Data and video segments saved.")
    os._exit(0)

atexit.register(cleanup)

def get_live_url(page_url):
    try:
        session = streamlink.Streamlink()
        streams = session.streams(page_url)
        return streams['480p'].url if '480p' in streams else streams['best'].url
    except Exception: 
        return None

def log_to_csv(cam_id, total, in_zone):
    """Saves detection data to a static CSV file."""
    log_file = f"log_cam_{cam_id}.csv"
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a') as f:
        if not file_exists:
            f.write("Timestamp,Total_Persons,Persons_In_Zone\n")
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{timestamp},{total},{in_zone}\n")

def ai_processor(page_url, cam_id, shared_model):
    live_url = get_live_url(page_url)
    if not live_url: return

    target_w, target_h = CAM_CONFIG[cam_id]["res"]
    zone_pts = CAM_CONFIG[cam_id]["zone"]
    
    cap = cv2.VideoCapture(live_url)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    segment_count = 1
    last_segment_time = time.time()
    
    video_writers[cam_id] = cv2.VideoWriter(
        f'cam_{cam_id}_part{segment_count}.mp4', fourcc, FPS, (target_w * 2, target_h)
    )

    print(f"[ONLINE] Cam {cam_id} Monitoring Active.")

    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            cap = cv2.VideoCapture(get_live_url(page_url))
            continue

        processed = cv2.resize(frame, (target_w, target_h))
        before_frame = processed.copy()

        try:
            with gpu_lock:
                results = shared_model.predict(
                    processed, imgsz=640, verbose=False, 
                    device=DEVICE, half=True, conf=0.45, classes=TARGET_CLASSES
                )
            
            after_frame = results[0].plot()
            boxes = results[0].boxes.xyxy.cpu().numpy()
            
            # --- COUNTER LOGIC ---
            total_persons = len(boxes)
            persons_in_zone = 0

            for box in boxes:
                cx, cy = int((box[0] + box[2]) / 2), int(box[3])
                if cv2.pointPolygonTest(zone_pts, (cx, cy), False) >= 0:
                    persons_in_zone += 1
                    cv2.circle(after_frame, (cx, cy), 12, (0, 0, 255), -1)

            intrusion_detected = persons_in_zone > 0
            
            # --- DATA LOGGING ---
            log_to_csv(cam_id, total_persons, persons_in_zone)

            # --- DASHBOARD OVERLAY ---
            # Semi-transparent UI box
            overlay = after_frame.copy()
            cv2.rectangle(overlay, (5, 5), (280, 110), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.5, after_frame, 0.5, 0, after_frame)
            
            # Text Stats
            cv2.putText(after_frame, f"TOTAL PERSONS: {total_persons}", (15, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            z_color = (0, 0, 255) if intrusion_detected else (0, 255, 0)
            cv2.putText(after_frame, f"IN ZONE: {persons_in_zone}", (15, 80), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, z_color, 2)

            # Alert Banner
            zone_color = (0, 0, 255) if intrusion_detected else (0, 255, 0)
            cv2.polylines(after_frame, [zone_pts], True, zone_color, 3)
            
            combined = np.hstack((before_frame, after_frame))
            video_writers[cam_id].write(combined)

            if time.time() - last_segment_time > SEGMENT_DURATION:
                video_writers[cam_id].release()
                segment_count += 1
                video_writers[cam_id] = cv2.VideoWriter(
                    f'cam_{cam_id}_part{segment_count}.mp4', fourcc, FPS, (target_w * 2, target_h)
                )
                last_segment_time = time.time()

            # Terminal Output
            log_time = time.strftime("%H:%M:%S")
            sys.stdout.write(f"[{log_time}] CAM {cam_id}: Total={total_persons} | InZone={persons_in_zone}\n")
            sys.stdout.flush()

        except Exception as e:
            continue
        
        time.sleep(1/FPS)

    cap.release()

if __name__ == "__main__":
    try:
        print("--- SecureZone AI Monitor: ACTIVE ---")
        main_model = YOLO(MODEL_NAME).to(DEVICE)
        main_model.fuse() 
        
        t0 = threading.Thread(target=ai_processor, args=(URLS[0], 0, main_model), daemon=True)
        t1 = threading.Thread(target=ai_processor, args=(URLS[1], 1, main_model), daemon=True)
        
        t0.start()
        time.sleep(2)
        t1.start()

        while True: time.sleep(1)
    except KeyboardInterrupt: 
        cleanup()

In [ ]:
import cv2
import time
import os
import torch
import streamlink
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading
import sys
import base64
import json
from ultralytics import YOLO
from datetime import datetime

# --- CONFIGURATION ---
MODEL_NAME = 'yolo11m.pt'
DEVICE = 'cuda:0'
FPS = 15.0
SEGMENT_DURATION = 300 
STATIC_ROOT = "Static_Storage"

# Aapke naye Coordinates
ZONE_0 = np.array([[129, 399], [272, 360], [452, 461], [160, 522]], dtype=np.int32)
ZONE_1 = np.array([[381, 273], [546, 407], [732, 280], [600, 223]], dtype=np.int32)

CAMS_DATA = {
    0: {
        "name": "Dublin", 
        "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", 
        "res": (955, 542), # testimg0.png dimensions
        "zone": ZONE_0
    },
    1: {
        "name": "NYC", 
        "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", 
        "res": (955, 537), # testimg1.png dimensions
        "zone": ZONE_1
    }
}

gpu_lock = threading.Lock()
stop_now = False

def get_stream_url(cam_url):
    try:
        session = streamlink.Streamlink()
        session.set_option("http-header", [("User-Agent", "Mozilla/5.0")])
        streams = session.streams(cam_url)
        return streams['720p'].url if '720p' in streams else streams['best'].url
    except Exception:
        return None

def process_camera(cam_id):
    global stop_now
    config = CAMS_DATA[cam_id]
    cam_name = config["name"]
    tw, th = config["res"]
    
    # --- FOLDER SETUP ---
    cam_root = os.path.join(STATIC_ROOT, cam_name)
    paths = {
        "videos": os.path.join(cam_root, "Videos"),
        "captures": os.path.join(cam_root, "Captures"),
        "snapshots": os.path.join(cam_root, "Snapshots"),
        "logs": os.path.join(cam_root, "Logs")
    }
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    model = YOLO(MODEL_NAME).to(DEVICE)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    
    writer = None
    last_seg_time = 0
    seg_count = 1
    current_video_name = "None"
    reference_saved = False

    while not stop_now:
        print(f"[{cam_name}] Connecting at {tw}x{th}...")
        stream_url = get_stream_url(config["url"])
        
        if not stream_url:
            time.sleep(5); continue

        os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "timeout;5000000|reconnect;1"
        cap = cv2.VideoCapture(stream_url, cv2.CAP_FFMPEG)

        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            # Resizing to your detected image dimensions
            frame = cv2.resize(frame, (tw, th))

            if not reference_saved:
                ref_path = os.path.join(paths["snapshots"], "reference_zone.jpg")
                ref_frame = frame.copy()
                cv2.polylines(ref_frame, [config["zone"]], True, (255, 255, 0), 2)
                cv2.imwrite(ref_path, ref_frame)
                reference_saved = True
            
            with gpu_lock:
                results = model.predict(frame, imgsz=max(tw, th), verbose=False, device=DEVICE, half=True, classes=[0])
            
            after_frame = frame.copy()
            boxes = results[0].boxes.xyxy.cpu().numpy()
            in_zone_count = 0
            
            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                cx, cy = int((x1+x2)/2), y2
                if cv2.pointPolygonTest(config["zone"], (cx, cy), False) >= 0:
                    in_zone_count += 1
                    cv2.rectangle(after_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                else:
                    cv2.rectangle(after_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Latest Snapshot Update
            latest_path = os.path.join(paths["snapshots"], "latest_event.jpg")
            cv2.polylines(after_frame, [config["zone"]], True, (0, 0, 255) if in_zone_count > 0 else (255, 255, 255), 2)
            cv2.imwrite(latest_path, after_frame)

            # JSON Logging logic
            status = "CRITICAL" if in_zone_count > 0 else "NORMAL"
            intruder_b64 = "NONE"
            if status == "CRITICAL":
                _, buffer = cv2.imencode('.jpg', after_frame)
                intruder_b64 = base64.b64encode(buffer).decode('utf-8')

            json_output = {
                "status": status,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "count": {"total": len(boxes), "in_zone": in_zone_count, "outside_zone": len(boxes)-in_zone_count},
                "visuals": {
                    "reference_snapshot": f"{cam_name}/Snapshots/reference_zone.jpg",
                    "current_snapshot": f"{cam_name}/Snapshots/latest_event.jpg",
                    "intruder_capture": intruder_b64,
                    "video_segment": current_video_name
                }
            }

            with open(os.path.join(paths["logs"], "current_state.json"), 'w') as f:
                json.dump({cam_name: json_output}, f, indent=2)

            # Video Writing (tw*2 because of horizontal stack)
            now = time.time()
            if writer is None or (now - last_seg_time > SEGMENT_DURATION):
                if writer: writer.release()
                current_video_name = f"{cam_name}_Seg_{seg_count}_{datetime.now().strftime('%H%M%S')}.mp4"
                writer = cv2.VideoWriter(os.path.join(paths["videos"], current_video_name), fourcc, FPS, (tw * 2, th))
                last_seg_time = now
                seg_count += 1

            combined = np.hstack((frame, after_frame))
            writer.write(combined)
            
            sys.stdout.write(f"\r[{cam_name}] {tw}x{th} | Persons: {len(boxes)} | InZone: {in_zone_count}")
            sys.stdout.flush()

        cap.release()
        time.sleep(2)

    if writer: writer.release()

if __name__ == "__main__":
    with ThreadPoolExecutor(max_workers=2) as executor:
        try:
            executor.map(process_camera, [0, 1])
        except KeyboardInterrupt:
            stop_now = True
            print("\nStopping...")

In [ ]:
import cv2
import time
import os
import torch
import streamlink
import numpy as np
import threading
import sys
import json
from ultralytics import YOLO
from datetime import datetime

# --- SETTINGS ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = "Static_Storage"

# Updated Coordinates
CAMS_DATA = {
    0: {
        "name": "Dublin", 
        "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", 
        "res": (955, 542), 
        "zone": np.array([[137, 359], [142, 480], [285, 374], [139, 351]], dtype=np.int32)
    },
    1: {
        "name": "NYC", 
        "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", 
        "res": (955, 537), 
        "zone": np.array([[404, 272], [554, 395], [598, 282], [407, 274]], dtype=np.int32)
    }
}

stop_now = False
gpu_lock = threading.Lock()
stats = {0: "Initializing...", 1: "Initializing..."}
tracking_db = {0: {}, 1: {}}

def get_url(cam_url):
    try:
        session = streamlink.Streamlink()
        session.set_option("http-headers", {"Referer": "https://www.earthcam.com/"})
        streams = session.streams(cam_url)
        return streams['720p'].url if '720p' in streams else streams['best'].url
    except: return None

def process_camera(cam_id):
    global stop_now, stats, tracking_db
    cfg = CAMS_DATA[cam_id]
    name, tw, th = cfg["name"], cfg["res"][0], cfg["res"][1]
    
    # Folders Management
    cam_root = os.path.join(STATIC_ROOT, name)
    paths = {k: os.path.join(cam_root, v) for k,v in {"videos":"Videos", "captures":"Captures", "snapshots":"Snapshots", "logs":"Logs"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    writer = None
    last_seg_time = 0
    zone_snap_saved = False # Flag to force save once per run

    while not stop_now:
        url = get_url(cfg["url"])
        if not url: 
            stats[cam_id] = "Conn Error"
            time.sleep(2); continue
        
        cap = cv2.VideoCapture(url, cv2.CAP_FFMPEG)
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret or stop_now: break 
            
            frame = cv2.resize(frame, (tw, th))
            before_frame = frame.copy()
            after_frame = frame.copy()

            # --- SNAPSHOT FIX: Har restart par naya snapshot lega ---
            if not zone_snap_saved:
                snap_path = os.path.join(paths["snapshots"], "zone_view.jpg")
                snap_frame = after_frame.copy()
                cv2.polylines(snap_frame, [cfg["zone"]], True, (255, 255, 0), 2)
                cv2.imwrite(snap_path, snap_frame)
                zone_snap_saved = True

            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.4, verbose=False, device=DEVICE, classes=[0])
            
            in_zone_count = 0
            current_ids = []
            
            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)
                
                # Check In-Zone
                for b in boxes:
                    if cv2.pointPolygonTest(cfg["zone"], (int((b[0]+b[2])/2), b[3]), False) >= 0:
                        in_zone_count += 1

                # Dynamic Color
                z_clr = (0,0,255) if in_zone_count > 0 else (255,255,0)
                cv2.polylines(after_frame, [cfg["zone"]], True, z_clr, 3)

                for box, obj_id in zip(boxes, ids):
                    current_ids.append(obj_id)
                    x1, y1, x2, y2 = map(int, box)
                    if cv2.pointPolygonTest(cfg["zone"], (int((x1+x2)/2), y2), False) >= 0:
                        cv2.rectangle(after_frame, (x1,y1), (x2,y2), (0,0,255), 3)
                        if obj_id not in tracking_db[cam_id]:
                            ts = datetime.now().strftime("%H%M%S")
                            # Yahan 'after_frame' save ho raha hai with RED zone & box
                            cv2.imwrite(os.path.join(paths["captures"], f"ID_{obj_id}_{ts}.jpg"), after_frame)
                            tracking_db[cam_id][obj_id] = {"id": int(obj_id), "in": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
                    else:
                        cv2.rectangle(after_frame, (x1,y1), (x2,y2), (0,255,0), 2)
            else:
                cv2.polylines(after_frame, [cfg["zone"]], True, (255, 255, 0), 3)

            # Logs Management
            for sid in list(tracking_db[cam_id].keys()):
                if sid not in current_ids:
                    tracking_db[cam_id][sid]["out"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    log_file = os.path.join(paths["logs"], "activity_log.json")
                    with open(log_file, 'a') as f:
                        json.dump(tracking_db[cam_id][sid], f); f.write("\n")
                    del tracking_db[cam_id][sid]

            # Video Recording (Stable AVI Segments)
            now = time.time()
            if writer is None or (now - last_seg_time > 300):
                if writer: writer.release()
                v_name = f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.avi"
                v_path = os.path.join(paths["videos"], v_name)
                writer = cv2.VideoWriter(v_path, cv2.VideoWriter_fourcc(*'XVID'), 15.0, (tw*2, th))
                last_seg_time = now

            writer.write(np.hstack((before_frame, after_frame)))
            stats[cam_id] = f"Detected: {len(current_ids)} | InZone: {in_zone_count}"
            sys.stdout.write(f"\r[Dublin] {stats[0]} | [NYC] {stats[1]}      ")
            sys.stdout.flush()

        cap.release()
    if writer: writer.release()

if __name__ == "__main__":
    print("--- FULL MONITORING SYSTEM (STABLE & FAST SHUTDOWN) ---")
    
    threads = []
    for cam_id in [0, 1]:
        t = threading.Thread(target=process_camera, args=(cam_id,))
        t.daemon = True 
        t.start()
        threads.append(t)

    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nShutting down immediately...")
        stop_now = True
        sys.exit(0)

In [ ]:
import cv2
import time
import os
import torch
import streamlink
import numpy as np
import threading
import sys
import json
from ultralytics import YOLO
from datetime import datetime

# --- SETTINGS ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = "Static_Storage"

# Updated Coordinates & Zones
CAMS_DATA = {
    0: {
        "name": "Dublin", 
        "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", 
        "res": (955, 542), 
        "zone": np.array([[131, 405], [257, 335], [414, 432], [165, 538]], dtype=np.int32)
    },
    1: {
        "name": "NYC", 
        "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", 
        "res": (955, 537), 
        "zone": np.array([[455, 257], [620, 361], [716, 296], [599, 221]], dtype=np.int32)
    }
}

stop_now = False
gpu_lock = threading.Lock()
stats = {0: "Initializing...", 1: "Initializing..."}
tracking_db = {0: {}, 1: {}}

# --- HELPER: COLOR DETECTION ---
def get_color_name(crop):
    """Detects average color and maps to common names."""
    if crop is None or crop.size == 0: return "Unknown"
    avg_bgr = cv2.mean(crop)[:3]
    
    colors = {
        "White": [220, 220, 220], "Black": [30, 30, 30], "Gray": [128, 128, 128],
        "Red": [0, 0, 255], "Blue": [255, 0, 0], "Green": [0, 255, 0],
        "Yellow": [0, 255, 255], "Orange": [0, 165, 255]
    }
    
    distances = {name: np.linalg.norm(np.array(avg_bgr) - np.array(val)) for name, val in colors.items()}
    return min(distances, key=distances.get)

def get_url(cam_url):
    try:
        session = streamlink.Streamlink()
        session.set_option("http-headers", {"Referer": "https://www.earthcam.com/"})
        streams = session.streams(cam_url)
        return streams['720p'].url if '720p' in streams else streams['best'].url
    except: return None

# --- CORE PROCESS ---
def process_camera(cam_id):
    global stop_now, stats, tracking_db
    cfg = CAMS_DATA[cam_id]
    name, tw, th = cfg["name"], cfg["res"][0], cfg["res"][1]
    
    # Folders Management
    cam_root = os.path.join(STATIC_ROOT, name)
    paths = {k: os.path.join(cam_root, v) for k,v in {"videos":"Videos", "captures":"Captures", "snapshots":"Snapshots", "logs":"Logs"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    writer = None
    last_seg_time = 0
    zone_snap_saved = False 

    while not stop_now:
        url = get_url(cfg["url"])
        if not url: 
            stats[cam_id] = "Conn Error"
            time.sleep(2); continue
        
        cap = cv2.VideoCapture(url, cv2.CAP_FFMPEG)
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret or stop_now: break 
            
            frame = cv2.resize(frame, (tw, th))
            before_frame = frame.copy() # Clean copy (NO DRAWING) for logic/snapshots
            after_frame = frame.copy()  # Visual copy for Bounding Boxes

            # --- SNAPSHOT FIX: Use before_frame to avoid drawing bugs ---
            if not zone_snap_saved:
                snap_path = os.path.join(paths["snapshots"], "zone_view.jpg")
                snap_frame = before_frame.copy()
                cv2.polylines(snap_frame, [cfg["zone"]], True, (255, 255, 0), 2)
                cv2.imwrite(snap_path, snap_frame)
                zone_snap_saved = True

            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.4, verbose=False, device=DEVICE, classes=[0])
            
            in_zone_count = 0
            current_ids = []
            
            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)
                
                # Dynamic Zone Color check
                for b in boxes:
                    if cv2.pointPolygonTest(cfg["zone"], (int((b[0]+b[2])/2), b[3]), False) >= 0:
                        in_zone_count += 1

                z_clr = (0,0,255) if in_zone_count > 0 else (255,255,0)
                cv2.polylines(after_frame, [cfg["zone"]], True, z_clr, 3)

                for box, obj_id in zip(boxes, ids):
                    current_ids.append(obj_id)
                    x1, y1, x2, y2 = map(int, box)
                    
                    if cv2.pointPolygonTest(cfg["zone"], (int((x1+x2)/2), y2), False) >= 0:
                        cv2.rectangle(after_frame, (x1,y1), (x2,y2), (0,0,255), 3)
                        
                        # LOG ENTRY (Triggered only once per person)
                        if obj_id not in tracking_db[cam_id]:
                            # Color Extraction from the CLEAN frame
                            person_crop = before_frame[max(0,y1):min(th,y2), max(0,x1):min(tw,x2)]
                            ph, pw = person_crop.shape[:2]
                            shirt, pant = "Unknown", "Unknown"
                            if ph > 30:
                                shirt = get_color_name(person_crop[0:int(ph*0.45), :])
                                pant = get_color_name(person_crop[int(ph*0.55):ph, :])

                            # Image Capture
                            ts = datetime.now().strftime("%H%M%S")
                            photo_path = os.path.join(paths["captures"], f"ID_{obj_id}_{ts}.jpg")
                            cv2.imwrite(photo_path, after_frame)
                            
                            # JSON History Entry
                            tracking_db[cam_id][obj_id] = {
                                "person_id": int(obj_id),
                                "photo": photo_path,
                                "shirt": shirt,
                                "pant": pant,
                                "time_in": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                "time_out": "Still Inside"
                            }
                    else:
                        cv2.rectangle(after_frame, (x1,y1), (x2,y2), (0,255,0), 2)
            else:
                cv2.polylines(after_frame, [cfg["zone"]], True, (255, 255, 0), 3)

            # --- EXIT HANDLING (Writes Old -> New) ---
            for sid in list(tracking_db[cam_id].keys()):
                if sid not in current_ids:
                    tracking_db[cam_id][sid]["time_out"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    log_file = os.path.join(paths["logs"], "activity_log.json")
                    with open(log_file, 'a') as f:
                        json.dump(tracking_db[cam_id][sid], f); f.write("\n")
                    del tracking_db[cam_id][sid]

            # Video Recording
            now = time.time()
            if writer is None or (now - last_seg_time > 300):
                if writer: writer.release()
                v_name = f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.avi"
                v_path = os.path.join(paths["videos"], v_name)
                writer = cv2.VideoWriter(v_path, cv2.VideoWriter_fourcc(*'XVID'), 15.0, (tw*2, th))
                last_seg_time = now

            writer.write(np.hstack((before_frame, after_frame)))
            stats[cam_id] = f"Detected: {len(current_ids)} | InZone: {in_zone_count}"
            sys.stdout.write(f"\r[Dublin] {stats[0]} | [NYC] {stats[1]}      ")
            sys.stdout.flush()

        cap.release()
    if writer: writer.release()

if __name__ == "__main__":
    print("--- SYSTEM STARTING ---")
    threads = []
    for cam_id in [0, 1]:
        t = threading.Thread(target=process_camera, args=(cam_id,))
        t.daemon = True 
        t.start()
        threads.append(t)

    try:
        while True: time.sleep(1)
    except KeyboardInterrupt:
        stop_now = True
        sys.exit(0)

# **OPEN REMOTE**

In [ ]:
import cv2
import time
import os
import streamlink
import numpy as np
import threading
import json
import requests
import urllib3
from ultralytics import YOLO
from datetime import datetime
from collections import deque

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = os.path.abspath("Static_Storage")

# Logic Parameters
EXIT_THRESHOLD = 8.0   
PRE_ROLL_SECONDS = 5    
PADDING_RATIO = 0.25   
FPS = 10.0 # Standardized for VideoWriter

# --- OPENREMOTE CONFIG ---
OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"
OR_ASSET_ID = "6RY5HlP2bSmn3UXwM7PAuX"

tracking_db = {0: {}, 1: {}}
lifetime_counts = {0: 0, 1: 0}
track_history = {0: set(), 1: set()} 
zone_snapshot_paths = {0: "", 1: ""}

gpu_lock = threading.Lock()
stop_now = False

CAMS_DATA = {
    0: {"name": "Dublin", "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", "res": (955, 542), "zone": np.array([[131, 405], [257, 335], [414, 432], [165, 538]], dtype=np.int32)},
    1: {"name": "NYC", "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", "res": (955, 537), "zone": np.array([[455, 257], [620, 361], [716, 296], [599, 221]], dtype=np.int32)}
}

# --- UTILS ---
def get_padded_crop(frame, box, pad_ratio, tw, th):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    px, py = int(w * pad_ratio), int(h * pad_ratio)
    nx1, ny1 = max(0, x1 - px), max(0, y1 - py)
    nx2, ny2 = min(tw, x2 + px), min(th, y2 + py)
    return frame[ny1:ny2, nx1:nx2]

def sync_to_openremote(data_dict):
    def run():
        try:
            r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                              data={"grant_type": "client_credentials", "client_id": OR_CLIENT_ID, "client_secret": OR_SECRET}, verify=False, timeout=3)
            token = r.json().get("access_token")
            if token:
                payload = [{"ref": {"id": OR_ASSET_ID, "name": "Data"}, "value": {k: v for k, v in data_dict.items() if not k.startswith('_')}}]
                requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", json=payload, headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=3)
        except: pass
    threading.Thread(target=run, daemon=True).start()

# --- CORE ---
def process_camera(cam_id):
    global stop_now, tracking_db, lifetime_counts, track_history
    cfg = CAMS_DATA[cam_id]
    name, tw, th, zone_pts = cfg["name"], cfg["res"][0], cfg["res"][1], cfg["zone"]
    
    paths = {k: os.path.join(STATIC_ROOT, name, v) for k,v in {"vids":"Videos", "caps":"Captures", "zones":"ZoneSnapshots"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    # Buffer saves "Combined" side-by-side frames
    frame_buffer = deque(maxlen=int(FPS * PRE_ROLL_SECONDS))
    writer, curr_vid_path, last_seg_time = None, "", 0
    zone_saved = False

    while not stop_now:
        try:
            streams = streamlink.Streamlink().streams(cfg["url"])
            cap = cv2.VideoCapture(streams['720p'].url if '720p' in streams else streams['best'].url)
        except: time.sleep(2); continue
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break 
            frame = cv2.resize(frame, (tw, th))
            raw_frame = frame.copy() # Left side plain frame
            curr_time = time.time()
            
            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.25, tracker="bytetrack.yaml", verbose=False, device=DEVICE, classes=[0])
            
            annotated_frame = frame.copy() # Right side with detection
            in_zone_ids_now = []

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)

                for box, obj_id in zip(boxes, ids):
                    x1, y1, x2, y2 = map(int, box)
                    if cv2.pointPolygonTest(zone_pts, (int((x1+x2)/2), y2), False) >= 0:
                        in_zone_ids_now.append(obj_id)
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                        
                        if obj_id in tracking_db[cam_id]:
                            tracking_db[cam_id][obj_id]["_last_seen"] = curr_time
                        else:
                            if obj_id not in track_history[cam_id]:
                                lifetime_counts[cam_id] += 1
                                track_history[cam_id].add(obj_id)
                            
                            ts = datetime.now().strftime("%H%M%S")
                            cp = os.path.join(paths["caps"], f"I_{obj_id}_{ts}.jpg")
                            wide_crop = get_padded_crop(frame, (x1, y1, x2, y2), PADDING_RATIO, tw, th)
                            cv2.imwrite(cp, wide_crop)

                            tracking_db[cam_id][obj_id] = {
                                "person_id": int(obj_id), "camera": name,
                                "time_in": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                "time_out": "Active", "zone_coords": zone_pts.tolist(),
                                "intruder_img_url": cp, "zone_snap_url": zone_snapshot_paths[cam_id],
                                "video_url": curr_vid_path, "_last_seen": curr_time
                            }
                            sync_to_openremote(tracking_db[cam_id][obj_id])

            # Zone Visuals
            zone_color = (0, 0, 255) if in_zone_ids_now else (0, 255, 255)
            cv2.polylines(annotated_frame, [zone_pts], True, zone_color, 3)

            # --- CREATE SIDE-BY-SIDE VIEW ---
            # np.hstack joins frames horizontally (Raw | Annotated)
            combined_frame = np.hstack((raw_frame, annotated_frame))

            if not zone_saved:
                z_path = os.path.join(paths["zones"], "Static_Config.jpg")
                cv2.imwrite(z_path, annotated_frame)
                zone_snapshot_paths[cam_id] = z_path
                zone_saved = True

            # --- SMART VIDEO WRITING ---
            if writer is None or (time.time() - last_seg_time > 300):
                if writer: writer.release()
                curr_vid_path = os.path.join(paths["vids"], f"SBS_{name}_{datetime.now().strftime('%H%M%S')}.avi")
                # Important: Width is tw * 2 because of hstack
                writer = cv2.VideoWriter(curr_vid_path, cv2.VideoWriter_fourcc(*'XVID'), FPS, (tw * 2, th))
                last_seg_time = time.time()
                # Write "Before" buffer frames
                for f in frame_buffer: writer.write(f)
            
            writer.write(combined_frame)
            frame_buffer.append(combined_frame)

            # Exit logic
            for sid in list(tracking_db[cam_id].keys()):
                if sid not in in_zone_ids_now and (curr_time - tracking_db[cam_id][sid]["_last_seen"] > EXIT_THRESHOLD):
                    tracking_db[cam_id][sid]["time_out"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    sync_to_openremote(tracking_db[cam_id][sid])
                    del tracking_db[cam_id][sid]
        cap.release()

# --- DASHBOARD ---
def print_dashboard():
    while not stop_now:
        os.system('cls' if os.name == 'nt' else 'clear')
        print("="*80)
        print(f" SECURITY HUB (SIDE-BY-SIDE MODE) | {datetime.now().strftime('%H:%M:%S')} ")
        print("="*80)
        for cid, cfg in CAMS_DATA.items():
            status = f"!!! ALERT ({len(tracking_db[cid])}) !!!" if tracking_db[cid] else "SECURE"
            print(f"{cfg['name']:<15} | {status:<20} | Lifetime Total: {lifetime_counts[cid]}")
        print("="*80)
        time.sleep(0.5)

if __name__ == "__main__":
    threading.Thread(target=print_dashboard, daemon=True).start()
    for cid in CAMS_DATA.keys():
        threading.Thread(target=process_camera, args=(cid,), daemon=True).start()
    try:
        while True: time.sleep(1)
    except KeyboardInterrupt:
        stop_now = True

# **FINAL CODE**

In [ ]:
import cv2
import time
import os
import streamlink
import numpy as np
import threading
import json
import requests
import urllib3
from ultralytics import YOLO
from datetime import datetime
from collections import deque

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = os.path.abspath("Static_Storage")

# Logic Parameters
EXIT_THRESHOLD = 8.0   
PRE_ROLL_SECONDS = 5    
PADDING_RATIO = 0.25   
FPS = 10.0 

# --- OPENREMOTE CONFIG ---
OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"
OR_ASSET_ID = "6RY5HlP2bSmn3UXwM7PAuX"

tracking_db = {0: {}, 1: {}}
lifetime_counts = {0: 0, 1: 0}
track_history = {0: set(), 1: set()} 
zone_snapshot_paths = {0: "", 1: ""}

gpu_lock = threading.Lock()
stop_now = False

CAMS_DATA = {
    0: {"name": "Dublin", "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", "res": (955, 542), "zone": np.array([[411, 398], [465, 470], [301, 539], [137, 435]], dtype=np.int32)},
    1: {"name": "NYC", "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", "res": (955, 537), "zone": np.array([[653, 424], [525, 244], [668, 197], [816, 333]], dtype=np.int32)}
}

# --- UTILS ---
def get_padded_crop(frame, box, pad_ratio, tw, th):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    px, py = int(w * pad_ratio), int(h * pad_ratio)
    nx1, ny1 = max(0, x1 - px), max(0, y1 - py)
    nx2, ny2 = min(tw, x2 + px), min(th, y2 + py)
    return frame[ny1:ny2, nx1:nx2]

def sync_to_openremote(data_dict):
    def run():
        try:
            r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                              data={"grant_type": "client_credentials", "client_id": OR_CLIENT_ID, "client_secret": OR_SECRET}, verify=False, timeout=3)
            token = r.json().get("access_token")
            if token:
                payload = [{"ref": {"id": OR_ASSET_ID, "name": "Data"}, "value": {k: v for k, v in data_dict.items() if not k.startswith('_')}}]
                requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", json=payload, headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=3)
        except: pass
    threading.Thread(target=run, daemon=True).start()

# --- CORE ---
def process_camera(cam_id):
    global stop_now, tracking_db, lifetime_counts, track_history
    cfg = CAMS_DATA[cam_id]
    name, tw, th, zone_pts = cfg["name"], cfg["res"][0], cfg["res"][1], cfg["zone"]
    
    paths = {k: os.path.join(STATIC_ROOT, name, v) for k,v in {"vids":"Videos", "caps":"Captures", "zones":"ZoneSnapshots"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    frame_buffer = deque(maxlen=int(FPS * PRE_ROLL_SECONDS))
    writer, curr_vid_path, last_seg_time = None, "", 0
    zone_saved = False

    while not stop_now:
        try:
            streams = streamlink.Streamlink().streams(cfg["url"])
            cap = cv2.VideoCapture(streams['720p'].url if '720p' in streams else streams['best'].url)
        except: time.sleep(2); continue
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break 
            frame = cv2.resize(frame, (tw, th))
            raw_frame = frame.copy()
            curr_time = time.time()
            
            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.25, tracker="bytetrack.yaml", verbose=False, device=DEVICE, classes=[0])
            
            annotated_frame = frame.copy()
            in_zone_ids_now = []

            # --- ZONE SNAPSHOT (HISTORY VERSION) ---
            # Save a unique zone map for every script run to maintain history
            if not zone_saved:
                ts_zone = datetime.now().strftime("%Y%m%d_%H%M%S")
                z_path = os.path.join(paths["zones"], f"Zone_Config_{ts_zone}.jpg")
                
                # Draw zone on a fresh frame for record
                zone_preview = frame.copy()
                cv2.polylines(zone_preview, [zone_pts], True, (0, 255, 255), 3)
                cv2.imwrite(z_path, zone_preview)
                
                zone_snapshot_paths[cam_id] = z_path
                zone_saved = True

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)

                for box, obj_id in zip(boxes, ids):
                    x1, y1, x2, y2 = map(int, box)
                    if cv2.pointPolygonTest(zone_pts, (int((x1+x2)/2), y2), False) >= 0:
                        in_zone_ids_now.append(obj_id)
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                        
                        if obj_id in tracking_db[cam_id]:
                            tracking_db[cam_id][obj_id]["_last_seen"] = curr_time
                        else:
                            if obj_id not in track_history[cam_id]:
                                lifetime_counts[cam_id] += 1
                                track_history[cam_id].add(obj_id)
                            
                            ts = datetime.now().strftime("%H%M%S")
                            cp = os.path.join(paths["caps"], f"I_{obj_id}_{ts}.jpg")
                            wide_crop = get_padded_crop(frame, (x1, y1, x2, y2), PADDING_RATIO, tw, th)
                            cv2.imwrite(cp, wide_crop)

                            tracking_db[cam_id][obj_id] = {
                                "person_id": int(obj_id), "camera": name,
                                "time_in": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                "time_out": "Active", "zone_coords": zone_pts.tolist(),
                                "intruder_img_url": cp, 
                                "zone_snap_url": zone_snapshot_paths[cam_id], # Link to versioned zone
                                "video_url": curr_vid_path, "_last_seen": curr_time
                            }
                            sync_to_openremote(tracking_db[cam_id][obj_id])

            # Visuals
            zone_color = (0, 0, 255) if in_zone_ids_now else (0, 255, 255)
            cv2.polylines(annotated_frame, [zone_pts], True, zone_color, 3)
            combined_frame = np.hstack((raw_frame, annotated_frame))

            # Video Logic
            if writer is None or (time.time() - last_seg_time > 300):
                if writer: writer.release()
                curr_vid_path = os.path.join(paths["vids"], f"SBS_{name}_{datetime.now().strftime('%H%M%S')}.avi")
                writer = cv2.VideoWriter(curr_vid_path, cv2.VideoWriter_fourcc(*'XVID'), FPS, (tw * 2, th))
                last_seg_time = time.time()
                for f in frame_buffer: writer.write(f)
            
            writer.write(combined_frame)
            frame_buffer.append(combined_frame)

            for sid in list(tracking_db[cam_id].keys()):
                if sid not in in_zone_ids_now and (curr_time - tracking_db[cam_id][sid]["_last_seen"] > EXIT_THRESHOLD):
                    tracking_db[cam_id][sid]["time_out"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    sync_to_openremote(tracking_db[cam_id][sid])
                    del tracking_db[cam_id][sid]
        cap.release()

# --- DASHBOARD ---
def print_dashboard():
    while not stop_now:
        os.system('cls' if os.name == 'nt' else 'clear')
        print("="*80)
        print(f" SECURITY HUB (SBS + HISTORY MODE) | {datetime.now().strftime('%H:%M:%S')} ")
        print("="*80)
        for cid, cfg in CAMS_DATA.items():
            status = f"!!! ALERT ({len(tracking_db[cid])}) !!!" if tracking_db[cid] else "SECURE"
            print(f"{cfg['name']:<15} | {status:<20} | Lifetime Total: {lifetime_counts[cid]}")
            print(f"Active Zone File: {os.path.basename(zone_snapshot_paths[cid])}")
        print("="*80)
        time.sleep(0.5)

if __name__ == "__main__":
    threading.Thread(target=print_dashboard, daemon=True).start()
    for cid in CAMS_DATA.keys():
        threading.Thread(target=process_camera, args=(cid,), daemon=True).start()
    try:
        while True: time.sleep(1)
    except KeyboardInterrupt:
        stop_now = True

# **DIFFERENT ASSSETS IDS **

In [4]:
import cv2
import time
import os
import streamlink
import numpy as np
import threading
import json
import requests
import urllib3
from ultralytics import YOLO
from datetime import datetime
from collections import deque

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = os.path.abspath("Static_Storage")

# Logic Parameters
EXIT_THRESHOLD = 8.0   
PRE_ROLL_SECONDS = 5    
PADDING_RATIO = 0.45   
FPS = 10.0 

# Video Segment Tracking
segment_counters = {0: 1, 1: 1} 

# --- OPENREMOTE CONFIG ---
OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"

tracking_db = {0: {}, 1: {}}
lifetime_counts = {0: 0, 1: 0}
track_history = {0: set(), 1: set()} 
zone_snapshot_paths = {0: "", 1: ""}

gpu_lock = threading.Lock()
stop_now = False

CAMS_DATA = {
    0: {
        "name": "Dublin", 
        "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", 
        "asset_id": "7d2UcWwY76HUjMGYN4RgvV", 
        "res": (955, 542), 
        "zone": np.array([[134, 339], [254, 332], [319, 403], [132, 448]], dtype=np.int32)
    },
    1: {
        "name": "NYC", 
        "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", 
        "asset_id": "5u4zFNrPIP8GGNdhsxt2HU", 
        "res": (955, 537), 
        "zone": np.array([[120, 73], [646, 73], [646, 502], [120, 502]], dtype=np.int32)
    }
}

# --- UTILS ---
def get_padded_crop(frame, box, pad_ratio, tw, th):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    px, py = int(w * pad_ratio), int(h * pad_ratio)
    nx1, ny1 = max(0, x1 - px), max(0, y1 - py)
    nx2, ny2 = min(tw, x2 + px), min(th, y2 + py)
    crop = frame[ny1:ny2, nx1:nx2]
    if crop.size > 0:
        crop = cv2.resize(crop, None, fx=1.2, fy=1.2, interpolation=cv2.INTER_LANCZOS4)
    return crop

def sync_to_openremote(data_dict, asset_id):
    def run():
        try:
            r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                              data={"grant_type": "client_credentials", "client_id": OR_CLIENT_ID, "client_secret": OR_SECRET}, verify=False, timeout=3)
            token = r.json().get("access_token")
            if token:
                payload = [{"ref": {"id": asset_id, "name": "Data"}, "value": {k: v for k, v in data_dict.items() if not k.startswith('_')}}]
                requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", json=payload, headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=3)
        except: pass
    threading.Thread(target=run, daemon=True).start()

# --- CORE ---
def process_camera(cam_id):
    global stop_now, tracking_db, lifetime_counts, track_history, segment_counters
    cfg = CAMS_DATA[cam_id]
    name, tw, th, zone_pts, asset_id = cfg["name"], cfg["res"][0], cfg["res"][1], cfg["zone"], cfg["asset_id"]
    
    paths = {k: os.path.join(STATIC_ROOT, name, v) for k,v in {"vids":"Videos", "caps":"Captures", "zones":"ZoneSnapshots"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    frame_buffer = deque(maxlen=int(FPS * PRE_ROLL_SECONDS))
    writer, curr_vid_path, last_seg_time = None, "", 0
    zone_saved = False

    while not stop_now:
        try:
            streams = streamlink.Streamlink().streams(cfg["url"])
            cap = cv2.VideoCapture(streams['720p'].url if '720p' in streams else streams['best'].url)
        except: time.sleep(2); continue
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break 
            frame = cv2.resize(frame, (tw, th))
            raw_frame = frame.copy()
            curr_time = time.time()
            
            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.25, tracker="bytetrack.yaml", verbose=False, device=DEVICE, classes=[0])
            
            annotated_frame = frame.copy()
            in_zone_ids_now = []

            # Save Zone Snapshot once
            if not zone_saved:
                z_path = os.path.join(paths["zones"], f"Zone_Config_{name}.jpg")
                zone_preview = frame.copy()
                cv2.polylines(zone_preview, [zone_pts], True, (0, 255, 255), 3)
                cv2.imwrite(z_path, zone_preview)
                zone_snapshot_paths[cam_id] = z_path
                zone_saved = True

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)

                for box, obj_id in zip(boxes, ids):
                    x1, y1, x2, y2 = map(int, box)
                    feet_coord = (int((x1+x2)/2), y2)
                    is_intruder = cv2.pointPolygonTest(zone_pts, feet_coord, False) >= 0

                    if is_intruder:
                        in_zone_ids_now.append(obj_id)
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        
                        # --- PHASE 1: ENTRY (ID + Photo + Coords) ---
                        if obj_id in tracking_db[cam_id]:
                            tracking_db[cam_id][obj_id]["_last_seen"] = curr_time
                        else:
                            lifetime_counts[cam_id] += 1
                            ts = datetime.now().strftime("%H%M%S")
                            cp = os.path.join(paths["caps"], f"I_{obj_id}_{ts}.jpg")
                            wide_crop = get_padded_crop(frame, (x1, y1, x2, y2), PADDING_RATIO, tw, th)
                            cv2.imwrite(cp, wide_crop)

                            tracking_db[cam_id][obj_id] = {
                                "person_id": int(obj_id), 
                                "camera": name,
                                "time_in": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                "intruder_img_url": cp,
                                "zone_coords": zone_pts.tolist(), # Coordinates included here
                                "_last_seen": curr_time
                            }
                            sync_to_openremote(tracking_db[cam_id][obj_id], asset_id)
                    else:
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

            # --- VIDEO WRITING (Part 1, Part 2 Logic) ---
            if writer is None or (time.time() - last_seg_time > 300):
                if writer: writer.release()
                part_name = f"{name} Part {segment_counters[cam_id]}.avi"
                curr_vid_path = os.path.abspath(os.path.join(paths["vids"], part_name))
                writer = cv2.VideoWriter(curr_vid_path, cv2.VideoWriter_fourcc(*'XVID'), FPS, (tw * 2, th))
                last_seg_time = time.time()
                segment_counters[cam_id] += 1
                for f in frame_buffer: writer.write(f)
            
            combined_frame = np.hstack((raw_frame, annotated_frame))
            writer.write(combined_frame)
            frame_buffer.append(combined_frame)

            # --- PHASE 2: EXIT (Time Out + Video + Snap) ---
            for sid in list(tracking_db[cam_id].keys()):
                if sid not in in_zone_ids_now and (curr_time - tracking_db[cam_id][sid]["_last_seen"] > EXIT_THRESHOLD):
                    
                    tracking_db[cam_id][sid].update({
                        "time_out": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        "video_url": curr_vid_path,
                        "zone_snap_url": zone_snapshot_paths[cam_id]
                    })
                    
                    sync_to_openremote(tracking_db[cam_id][sid], asset_id)
                    del tracking_db[cam_id][sid]

        cap.release()

if __name__ == "__main__":
    for cid in CAMS_DATA.keys():
        threading.Thread(target=process_camera, args=(cid,), daemon=True).start()
    try:
        while True: time.sleep(1)
    except KeyboardInterrupt: stop_now = True

[ WARN:3@535.586] global cap.cpp:175 open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.12.0) /io/opencv/modules/videoio/src/cap_images.cpp:237: error: (-5:Bad argument) CAP_IMAGES: error, expected '0?[1-9][du]' pattern, got: https://videos-3.earthcam.com/fecnetwork/24322.flv/chunklist_w2071017929.m3u8?t=qAP3aum0UbcBtTuO%2Fx%2F7Lz9UytxcCWnrPDJyjgaIxermaKuSHKiLF6M1N15lZHHH&td=202601290511 in function 'icvExtractPattern'


[ WARN:3@537.593] global cap.cpp:175 open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.12.0) /io/opencv/modules/videoio/src/cap_images.cpp:237: error: (-5:Bad argument) CAP_IMAGES: error, expected '0?[1-9][du]' pattern, got: https://videos-3.earthcam.com/fecnetwork/24322.flv/chunklist_w121616879.m3u8?t=qAP3aum0UbcBtTuO%2Fx%2F7Lz9UytxcCWnrPDJyjgaIxeqgGbnpxhripTktUQJQD3De&td=202601290521 in function 'icvExtractPattern'


[ WARN:3@538.466] global cap.cpp:175 open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.12.0) /io/opencv/modules/videoio/src

# **Image snapshots** 

In [ ]:
import requests
import urllib3

# SSL Warnings ko disable karne ke liye
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"

CAMS_DATA = {
    0: {
        "name": "Dublin",
        "asset_id": "7d2UcWwY76HUjMGYN4RgvV",
        "img_path": "/kaggle/input/snapshots/testimg0.png",
        "res": (955, 542)
    },
    1: {
        "name": "NYC",
        "asset_id": "5u4zFNrPIP8GGNdhsxt2HU",
        "img_path": "/kaggle/input/snapshots/testimg1.png",
        "res": (955, 537)
    }
}

def sync_to_openremote():
    try:
        # 1. Get Access Token
        r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                          data={
                              "grant_type": "client_credentials", 
                              "client_id": OR_CLIENT_ID, 
                              "client_secret": OR_SECRET
                          }, 
                          verify=False, timeout=10)
        
        token = r.json().get("access_token")
        if not token:
            print("Token fetch failed!")
            return

        headers = {
            "Authorization": f"Bearer {token}", 
            "Content-Type": "application/json"
        }

        for cam_id, cfg in CAMS_DATA.items():
            # 2. Payload as per your requirement
            payload = {
                "Camera_Name": cfg["name"],
                "Zone_Image_Path": cfg["img_path"],
                "Resolution": f"{cfg['res'][0]}x{cfg['res'][1]}"
            }

            # 3. OpenRemote API Call (Updating the 'Data' attribute)
            attr_payload = [{
                "ref": {
                    "id": cfg["asset_id"], 
                    "name": "Data"
                }, 
                "value": payload
            }]
            
            resp = requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", 
                                json=attr_payload, headers=headers, verify=False)
            
            if resp.status_code in [200, 204]:
                print(f"✅ Synced: {cfg['name']} | Path: {cfg['img_path']}")
            else:
                print(f"❌ Failed for {cfg['name']}: {resp.status_code}")

    except Exception as e:
        print(f"Error occurred: {e}")

# Script ko run karein
if __name__ == "__main__":
    sync_to_openremote()